In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

df = pd.read_parquet("../compact_eq_design/outputs/socialfx_curves_40_23.parquet")

print("shape:", df.shape)
display(df.head(3))

shape: (1595, 10)


,id,text,lang,ratings_consistency,curve_40,curve_23,freqs_40,freqs_23,n_bands_40,n_bands_23
0,eq_0,hot,English,0.797209,"[-1.2872984409332275, -1.5329641103744507, -1....","[-1.2872984409332275, -1.5329641103744507, -1....","[20.0, 50.0, 83.0, 120.0, 161.0, 208.0, 259.0,...","[20.0, 50.0, 83.0, 120.0, 159.5102996826172, 2...",40,23
1,eq_1,wet,English,0.481343,"[-0.9765962958335876, -0.7818208336830139, -0....","[-0.9765962958335876, -0.7818208336830139, -0....","[20.0, 50.0, 83.0, 120.0, 161.0, 208.0, 259.0,...","[20.0, 50.0, 83.0, 120.0, 159.5102996826172, 2...",40,23
2,eq_10,warm,English,0.806225,"[1.970152497291565, 2.0636439323425293, 2.1096...","[1.970152497291565, 2.0636439323425293, 2.1096...","[20.0, 50.0, 83.0, 120.0, 161.0, 208.0, 259.0,...","[20.0, 50.0, 83.0, 120.0, 159.5102996826172, 2...",40,23


In [3]:
def to_np(x, dtype=np.float32):
    return np.asarray(x, dtype=dtype)

def stack_curves(series):
    return np.vstack(series.apply(lambda x: np.asarray(x, dtype=np.float32)).to_list())

# Проверка
curve40 = stack_curves(df["curve_40"])
curve23 = stack_curves(df["curve_23"])

freqs40 = to_np(df["freqs_40"].iloc[0], np.float32)
freqs23 = to_np(df["freqs_23"].iloc[0], np.float32)

print("curve40:", curve40.shape)
print("curve23:", curve23.shape)
print("freqs40:", freqs40.shape, freqs40[:5], "...", freqs40[-5:])
print("freqs23:", freqs23.shape, freqs23[:5], "...", freqs23[-5:])

curve40: (1595, 40)
curve23: (1595, 23)
freqs40: (40,) [ 20.  50.  83. 120. 161.] ... [12474. 13984. 15675. 17566. 19682.]
freqs23: (23,) [ 20.      50.      83.     120.     159.5103] ... [ 6934.2607  8568.995  12000.     14000.     16000.    ]


In [5]:
FEATURE_NAMES = ["bass", "tilt", "presence", "air", "lowmid", "sparkle"]

# Центры и ширины в октавах
ZONE_CONFIG = {
    "bass":     {"center_hz":  90.0,  "sigma_oct": 0.80},
    "lowmid":   {"center_hz": 280.0,  "sigma_oct": 0.65},
    "presence": {"center_hz": 3200.0, "sigma_oct": 0.60},
    "air":      {"center_hz": 9500.0, "sigma_oct": 0.75},
}

SPARKLE_START_HZ = 6500.0
SPARKLE_SLOPE_OCT = 0.55

In [6]:
def gaussian_weights(freqs_hz, center_hz, sigma_oct):
    freqs_hz = np.asarray(freqs_hz, dtype=np.float64)
    log_f = np.log2(freqs_hz)
    c = np.log2(float(center_hz))
    x = (log_f - c) / float(sigma_oct)
    w = np.exp(-0.5 * (x ** 2))
    s = w.sum()
    if s > 0:
        w = w / s
    return w.astype(np.float64)

def high_shelf_weights(freqs_hz, start_hz, slope_oct):
    freqs_hz = np.asarray(freqs_hz, dtype=np.float64)
    x = (np.log2(freqs_hz) - np.log2(float(start_hz))) / float(slope_oct)
    w = 1.0 / (1.0 + np.exp(-x))
    w = w - w.min()
    s = w.sum()
    if s > 0:
        w = w / s
    return w.astype(np.float64)

def tilt_weights(freqs_hz):
    freqs_hz = np.asarray(freqs_hz, dtype=np.float64)
    x = np.log2(freqs_hz)
    x = x - x.mean()
    denom = np.max(np.abs(x))
    if denom > 0:
        x = x / denom
    return x.astype(np.float64)

In [7]:
def curve_to_6d(curve_db, freqs_hz):
    curve_db = np.asarray(curve_db, dtype=np.float64)
    freqs_hz = np.asarray(freqs_hz, dtype=np.float64)

    # zone features
    bass_w = gaussian_weights(freqs_hz, **ZONE_CONFIG["bass"])
    lowmid_w = gaussian_weights(freqs_hz, **ZONE_CONFIG["lowmid"])
    presence_w = gaussian_weights(freqs_hz, **ZONE_CONFIG["presence"])
    air_w = gaussian_weights(freqs_hz, **ZONE_CONFIG["air"])
    sparkle_w = high_shelf_weights(freqs_hz, SPARKLE_START_HZ, SPARKLE_SLOPE_OCT)
    tilt_w = tilt_weights(freqs_hz)

    bass = float(np.sum(curve_db * bass_w))
    lowmid = float(np.sum(curve_db * lowmid_w))
    presence = float(np.sum(curve_db * presence_w))
    air = float(np.sum(curve_db * air_w))
    sparkle = float(np.sum(curve_db * sparkle_w))

    # tilt как взвешенный наклон
    tilt = float(np.sum(curve_db * tilt_w) / max(1e-8, np.sum(np.abs(tilt_w))))

    return np.array([bass, tilt, presence, air, lowmid, sparkle], dtype=np.float32)

In [8]:
z40 = np.vstack([curve_to_6d(c, freqs40) for c in curve40])
z23 = np.vstack([curve_to_6d(c, freqs23) for c in curve23])

print("z40:", z40.shape)
print("z23:", z23.shape)

z40_df = pd.DataFrame(z40, columns=[f"{name}_40_raw" for name in FEATURE_NAMES])
z23_df = pd.DataFrame(z23, columns=[f"{name}_23_raw" for name in FEATURE_NAMES])

display(z40_df.head(3))
display(z23_df.head(3))

z40: (1595, 6)
z23: (1595, 6)


,bass_40_raw,tilt_40_raw,presence_40_raw,air_40_raw,lowmid_40_raw,sparkle_40_raw
0,-1.287566,-0.006698,0.082135,-0.529829,0.033145,-0.597603
1,-0.725170,0.978346,-0.142970,1.207791,-0.890200,1.272812
2,1.904597,-0.901874,-0.095659,-0.585652,0.981721,-0.561150


,bass_23_raw,tilt_23_raw,presence_23_raw,air_23_raw,lowmid_23_raw,sparkle_23_raw
0,-1.290872,0.291944,0.124568,-0.571931,-0.102991,-0.602004
1,-0.726108,0.963652,-0.178811,1.406962,-0.913966,1.481533
2,1.906948,-1.133516,-0.103543,-0.669848,1.114597,-0.648133


In [9]:
def zscore_fit_transform(X):
    X = np.asarray(X, dtype=np.float64)
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std = np.where(std < 1e-8, 1.0, std)
    Xn = (X - mean) / std
    return Xn.astype(np.float32), mean.astype(np.float32), std.astype(np.float32)

z40_norm, z40_mean, z40_std = zscore_fit_transform(z40)
z23_norm, z23_mean, z23_std = zscore_fit_transform(z23)

z40n_df = pd.DataFrame(z40_norm, columns=[f"{name}_40" for name in FEATURE_NAMES])
z23n_df = pd.DataFrame(z23_norm, columns=[f"{name}_23" for name in FEATURE_NAMES])

display(z40n_df.head(3))
display(z23n_df.head(3))

,bass_40,tilt_40,presence_40,air_40,lowmid_40,sparkle_40
0,-1.126056,0.144874,0.113460,-0.665517,-0.123303,-0.773750
1,-0.689918,1.553454,-0.202264,1.941452,-1.056473,2.125769
2,1.349467,-1.135198,-0.135907,-0.749269,0.835368,-0.717241


,bass_23,tilt_23,presence_23,air_23,lowmid_23,sparkle_23
0,-1.127161,0.537807,0.174390,-0.684178,-0.256713,-0.730522
1,-0.689860,1.416019,-0.247524,2.180072,-1.032669,2.346461
2,1.348931,-1.325883,-0.142848,-0.825903,0.908299,-0.798645


In [10]:
df_out = df.copy()

for i, name in enumerate(FEATURE_NAMES):
    df_out[f"{name}_40_raw"] = z40[:, i]
    df_out[f"{name}_23_raw"] = z23[:, i]
    df_out[f"{name}_40"] = z40_norm[:, i]
    df_out[f"{name}_23"] = z23_norm[:, i]

display(df_out.head(3))
print("shape with features:", df_out.shape)

,id,text,lang,ratings_consistency,curve_40,curve_23,freqs_40,freqs_23,n_bands_40,n_bands_23,bass_40_raw,bass_23_raw,bass_40,bass_23,tilt_40_raw,tilt_23_raw,tilt_40,tilt_23,presence_40_raw,presence_23_raw,presence_40,presence_23,air_40_raw,air_23_raw,air_40,air_23,lowmid_40_raw,lowmid_23_raw,lowmid_40,lowmid_23,sparkle_40_raw,sparkle_23_raw,sparkle_40,sparkle_23
0,eq_0,hot,English,0.797209,"[-1.2872984409332275, -1.5329641103744507, -1....","[-1.2872984409332275, -1.5329641103744507, -1....","[20.0, 50.0, 83.0, 120.0, 161.0, 208.0, 259.0,...","[20.0, 50.0, 83.0, 120.0, 159.5102996826172, 2...",40,23,-1.287566,-1.290872,-1.126056,-1.127161,-0.006698,0.291944,0.144874,0.537807,0.082135,0.124568,0.113460,0.174390,-0.529829,-0.571931,-0.665517,-0.684178,0.033145,-0.102991,-0.123303,-0.256713,-0.597603,-0.602004,-0.773750,-0.730522
1,eq_1,wet,English,0.481343,"[-0.9765962958335876, -0.7818208336830139, -0....","[-0.9765962958335876, -0.7818208336830139, -0....","[20.0, 50.0, 83.0, 120.0, 161.0, 208.0, 259.0,...","[20.0, 50.0, 83.0, 120.0, 159.5102996826172, 2...",40,23,-0.725170,-0.726108,-0.689918,-0.689860,0.978346,0.963652,1.553454,1.416019,-0.142970,-0.178811,-0.202264,-0.247524,1.207791,1.406962,1.941452,2.180072,-0.890200,-0.913966,-1.056473,-1.032669,1.272812,1.481533,2.125769,2.346461
2,eq_10,warm,English,0.806225,"[1.970152497291565, 2.0636439323425293, 2.1096...","[1.970152497291565, 2.0636439323425293, 2.1096...","[20.0, 50.0, 83.0, 120.0, 161.0, 208.0, 259.0,...","[20.0, 50.0, 83.0, 120.0, 159.5102996826172, 2...",40,23,1.904597,1.906948,1.349467,1.348931,-0.901874,-1.133516,-1.135198,-1.325883,-0.095659,-0.103543,-0.135907,-0.142848,-0.585652,-0.669848,-0.749269,-0.825903,0.981721,1.114597,0.835368,0.908299,-0.561150,-0.648133,-0.717241,-0.798645


shape with features: (1595, 34)


In [11]:
rows = []
for i, name in enumerate(FEATURE_NAMES):
    abs_err = np.abs(z40_norm[:, i] - z23_norm[:, i])
    rows.append({
        "feature": name,
        "mae": float(abs_err.mean()),
        "std_abs_err": float(abs_err.std()),
        "max_abs_err": float(abs_err.max()),
    })

df_feature_error = pd.DataFrame(rows).sort_values("mae", ascending=False).reset_index(drop=True)
display(df_feature_error)

,feature,mae,std_abs_err,max_abs_err
0,tilt,0.168600,0.104911,0.453809
1,air,0.114909,0.086112,0.459528
2,sparkle,0.106869,0.084403,0.626073
3,presence,0.065084,0.047605,0.276333
4,lowmid,0.057802,0.039279,0.188726
5,bass,0.001793,0.001601,0.011880


In [18]:
from pathlib import Path

OUT_DIR = Path("./outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_FILE = OUT_DIR / "socialfx_curves_with_6d.parquet"
df_out.to_parquet(OUT_FILE, index=False)

print("saved to:", OUT_FILE.resolve())

saved to: C:\Users\makcc\PycharmProjects\EarLoop\research\mapper_v2\outputs\socialfx_curves_with_6d.parquet


In [17]:
norm_stats = pd.DataFrame({
    "feature": FEATURE_NAMES,
    "mean_40": z40_mean,
    "std_40": z40_std,
    "mean_23": z23_mean,
    "std_23": z23_std,
})

NORM_PATH = Path("./outputs/socialfx_6d_norm_stats.csv")
norm_stats.to_csv(NORM_PATH, index=False)

display(norm_stats)
print("saved to:", NORM_PATH.resolve())

,feature,mean_40,std_40,mean_23,std_23
0,bass,0.164473,1.289490,0.164832,1.291479
1,tilt,-0.108011,0.699317,-0.119403,0.764859
2,presence,0.001240,0.712980,-0.000828,0.719057
3,air,-0.086243,0.666529,-0.099237,0.690894
4,lowmid,0.155149,0.989470,0.165307,1.045130
5,sparkle,-0.098474,0.645078,-0.107341,0.677137


saved to: C:\Users\makcc\PycharmProjects\EarLoop\research\mapper_v2\outputs\socialfx_6d_norm_stats.csv
